In [ ]:
try:
  from vpop_calibration.api import *
except ImportError:
  !pip install vpop_calibration[full]

  print('Have to restart session. Please run again.')
  import os
  os.kill(os.getpid(), 9)

In [ ]:
!wget -q https://raw.githubusercontent.com/novainsilico/vpop-calibration/refs/heads/main/examples/benchmarking/Mavoglurant/CM_Mavoglurant.xml
!wget -q https://raw.githubusercontent.com/novainsilico/vpop-calibration/refs/heads/main/examples/benchmarking/Mavoglurant/Mavoglurant_Dataset.csv

In [ ]:
import pandas as pd
from plotnine import *
import numpy as np

## Uploading and preparing dataset

In [ ]:
df = pd.read_csv("Mavoglurant_Dataset.csv")

obs_df = (
    df.loc[df["EVID"] == 0]
    .rename(columns={"ID": "id", "TIME": "time", "DV": "value", "DOSE": "Dose"})[
        ["id", "time", "value", "Dose", "WT"]
    ]
    .astype({"value": "float"})
)
obs_df["time"] = obs_df["time"].apply(lambda t: t * 60 * 60)
obs_df["value"] = obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "logC15"
obs_df["protocol_arm"] = "identity"
display(df.head())
display(obs_df.head())

## Uploading computational model

In [ ]:
struct_model = StructuralSbml(
    model_path="CM_Mavoglurant.xml",
    inputs=["KbBR", "CLint", "KbMU", "KbAD", "KbBO", "KbRB", "Dose", "WT"],
    outputs=["logC15"],
)

struct_model.rr.setIntegrator("cvode")
integ = struct_model.rr.integrator
integ.setValue("stiff", True)
integ.setValue("relative_tolerance", 1e-6)
integ.setValue("absolute_tolerance", 1e-6)
integ.setValue("initial_time_step", 1e-6)

## Defining initial estimates


In [ ]:
input_params = {
    "model_intrinsic": {},
    "pdu": {
        "KbBR": {"prior": np.exp(1.1), "prior_omega": 0.5},
        "KbMU": {"prior": np.exp(0.3), "prior_omega": 0.5},
        "KbAD": {"prior": np.exp(2), "prior_omega": 0.5},
        "CLint": {"prior": np.exp(7.6), "prior_omega": 4},
        "KbBO": {"prior": np.exp(0.03), "prior_omega": 0.5},
        "KbRB": {"prior": np.exp(0.3), "prior_omega": 0.5},
    },
    "pdk": {"WT", "Dose"},
    "error_model": {
        "logC15": {"error_type": "additive", "initial_variance": 1},
    },
}

## Configuring SAEM

In [ ]:
config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=10,
        nb_iter_learning=90,
        nb_iter_smoothing=100,
        plot_frames=5,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)
nlme_model = NlmeModel(
    df=obs_df, input_params=input_params, structural_model=struct_model, config=config
)
nlme_model.initial_estimates.plot_distribution()

## SAEM start

In [ ]:
nlme_model.optimizer.run()

## Diagnostics

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(nb_samples=100)

In [ ]:
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")

In [ ]:
nlme_model.diagnostics.compute_population_residuals(nb_samples=100)
nlme_model.plot.weighted_residuals("pwres")

In [ ]:
nlme_model.plot.weighted_residuals("npde")

In [ ]:
nlme_model.diagnostics.compute_vpc(nb_samples=100)
nlme_model.plot.vpc()

In [ ]:
nlme_model.plot.conditional_codistributions()

In [ ]:
nlme_model.diagnostics.compute_log_likelihood_importance_sampling(
    nb_proposal_samples=100
)
print(
    f"Observed-data likelihood = {nlme_model.diagnostics.importance_sampler.log_lik:.2f}"
)

In [ ]:
nlme_model.diagnostics.compute_shrinkage(nb_samples=100)
parameter_names = nlme_model.diagnostics.model.pdu_names
shrinkage_percent = nlme_model.diagnostics.shrinkage.detach().cpu().numpy() * 100
shrinkage_df = pd.DataFrame(
    [shrinkage_percent],
    columns=parameter_names,
    index=["shrinkage(%)"],
)

display(shrinkage_df.round(1))

In [ ]:
conditional_samples = nlme_model.diagnostics.sampler.total_samples_parameters_df
vpop_df = conditional_samples.rename(columns={"id": "patientIndex"}).drop(
    columns=["id_ref"]
)
display(vpop_df)
# vpop.to_csv("vpop.csv", index=False)

In [ ]:
!pip install jinko-sdk
from jinko import JinkoClient

In [ ]:
FOLDER_ID = "383dc14a-a823-4d45-a560-8049ead5047e"
MODEL_SID = "cm-LFEt-i8FD"

client = JinkoClient()
client.auth_check()

In [ ]:
vpop = client.create_vpop_from_dataframe(
    df=vpop_df,
    name="SAEM vpop",
    description="SAEM conditional samples",
    folder=FOLDER_ID,
)
print(f"Vpop link: {vpop.url}")

In [ ]:
model = client.get_model(MODEL_SID)
trial_output_timeseries = model.create_simple_output_set(
    measures=["logC15"],
    name="Timeseries to output",
    folder=FOLDER_ID or None,
)
trial = model.create_trial(
    vpop=vpop,
    simple_output_set=trial_output_timeseries,
    name="Mavoglurant trial",
    folder=FOLDER_ID,
)
print(f"Trial link: {trial.url}")

In [ ]:
trial.run()
trial.wait_until_completed()

In [ ]:
visualization = trial.create_empty_trial_visualization(
    name="Trial visualization", folder=FOLDER_ID
)
print(f"Trial visualization link: {visualization.url}")